# 04 — Review Artifact Reader

Use this notebook to review one row without opening many files manually. It is the clean reviewer gateway for the concise row packet and browser-visible verdicts.

```text
output/<row_id>/
├── final_row.csv
├── review_summary.md
├── review_decision.json
├── candidate_decisions.csv
├── browser_visible_verdicts.json
├── browser_visible/
└── product_coding_input.json
```

```mermaid
flowchart TD
    A[Set ROW_ARTIFACT_DIR] --> B[Load review_decision.json]
    B --> C[Show selected URL]
    C --> D[Show gate checks]
    D --> E[Show browser-visible verdict]
    E --> F[Show candidate decisions]
    F --> G[Reviewer action]
```


In [ ]:
from pathlib import Path
import json

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROJECT_ROOT


## 1. Set row artifact path

Point this to one row folder generated by Notebook 01 or Notebook 02.


In [ ]:
ROW_ARTIFACT_DIR = PROJECT_ROOT / "output" / "PUT_ROW_ID_HERE"

review_json = ROW_ARTIFACT_DIR / "review_decision.json"
candidate_csv = ROW_ARTIFACT_DIR / "candidate_decisions.csv"
summary_md = ROW_ARTIFACT_DIR / "review_summary.md"
coding_json = ROW_ARTIFACT_DIR / "product_coding_input.json"
visible_json = ROW_ARTIFACT_DIR / "browser_visible_verdicts.json"
browser_dir = ROW_ARTIFACT_DIR / "browser_visible"

required = [review_json, candidate_csv, summary_md, coding_json]
optional = [visible_json, browser_dir]
status = pd.DataFrame({
    "artifact": [p.name for p in required + optional],
    "path": [str(p) for p in required + optional],
    "required": [True] * len(required) + [False] * len(optional),
    "exists": [p.exists() for p in required + optional],
})
display(status)

missing = [p for p in required if not p.exists()]
assert not missing, "Missing required concise review artifacts. Check ROW_ARTIFACT_DIR."


In [ ]:
payload = json.loads(review_json.read_text(encoding="utf-8"))
candidates = pd.read_csv(candidate_csv, dtype=str)
decision = payload.get("decision", {})
checks = payload.get("checks", {})
decision


## 2. Decision snapshot


In [ ]:
pd.DataFrame([decision]).T.rename(columns={0: "value"})


## 3. Gate checks


In [ ]:
pd.DataFrame([checks]).T.rename(columns={0: "value"})


## 4. Browser-visible verdicts

Use this section when the URL opens but the page shown to the user may be wrong, rerouted, blocked, or not a product page.


In [ ]:
if visible_json.exists():
    visible = json.loads(visible_json.read_text(encoding="utf-8"))
    rows = []
    for url, verdict in visible.items():
        rows.append({
            "url": url,
            "status": verdict.get("status"),
            "page_type": verdict.get("browser_visible_page_type"),
            "user_visible_product_match": verdict.get("user_visible_product_match"),
            "survives_visible_check": verdict.get("champion_should_survive_visible_check"),
            "confidence": verdict.get("user_visible_content_confidence"),
            "rerouted": verdict.get("rerouted_or_not"),
            "screenshot_path": verdict.get("screenshot_path"),
        })
    display(pd.DataFrame(rows))
else:
    print("browser_visible_verdicts.json not found for this row")


In [ ]:
if browser_dir.exists():
    files = sorted(p for p in browser_dir.iterdir() if p.is_file())
    display(pd.DataFrame({"file": [p.name for p in files], "path": [str(p) for p in files]}).head(50))
else:
    print("browser_visible folder not found")


## 5. Why selected


In [ ]:
for point in payload.get("why_selected", []):
    print("-", point)


## 6. How decided


In [ ]:
for point in payload.get("how_decided", []):
    print("-", point)


## 7. Candidate decisions


In [ ]:
review_columns = [
    "rank", "selected", "decision", "url", "confidence", "validation_status",
    "identity_status", "exact_product_check", "country_check", "retailer_check",
    "scrapable", "product_page", "reason",
]
display(candidates[[c for c in review_columns if c in candidates.columns]])


## 8. Human-readable summary


In [ ]:
print(summary_md)
print("\nFirst 1200 characters:\n")
print(summary_md.read_text(encoding="utf-8")[:1200])


## 9. Product coding input


In [ ]:
coding_payload = json.loads(coding_json.read_text(encoding="utf-8"))
pd.DataFrame([coding_payload]).T.rename(columns={0: "value"}).head(60)
